# 변경사항 감지

새로 수집한 데이터와 이전 데이터를 비교하여 새로 추가된 필드만 감지합니다.

## Parameters

In [ ]:
# Papermill parameters (이전 단계에서 전달받음)
saved_file_path = ""
product_collection_result = {}
total_products_collected = 0
version_date = None

## 1. 라이브러리 및 설정 로드

In [ ]:
import json
import logging
import re
from datetime import datetime
from pathlib import Path
from typing import Dict, List, Any, Optional, Set, Tuple
import sys

# Add parent directory to path for config import
sys.path.append('..')
from config import config

# 로깅 설정
logging.basicConfig(level=getattr(logging, config.log_level.upper()))
logger = logging.getLogger(__name__)

print(f"이전 단계에서 받은 정보:")
print(f"- 저장된 파일: {saved_file_path}")
print(f"- 수집된 상품 수: {total_products_collected}")
print(f"- 버전 날짜: {version_date}")
print(f"- 수집 결과: {product_collection_result.get('collection_stats', {})}")

## 2. 파일 탐지 함수 정의

In [ ]:
def find_mobile_plan_files(base_dir: str) -> List[Tuple[str, str]]:
    """mobile_plan_info_*.json 파일들을 찾아서 날짜순으로 정렬"""
    path = Path(base_dir)
    if not path.exists():
        return []
    
    pattern = re.compile(r'^mobile_plan_info_(\d{8})\.json$')
    files = []
    
    for file_path in path.iterdir():
        if file_path.is_file():
            match = pattern.match(file_path.name)
            if match:
                date_str = match.group(1)
                files.append((date_str, file_path.name))
    
    # 날짜별로 정렬 (최신순)
    files.sort(key=lambda x: x[0], reverse=True)
    return files

def get_comparison_files(base_dir: str) -> Tuple[Optional[str], Optional[str]]:
    """비교할 파일 쌍 찾기 (이전, 현재)"""
    files = find_mobile_plan_files(base_dir)
    
    if len(files) < 2:
        return None, files[0][1] if files else None
    
    return files[1][1], files[0][1]  # (두 번째 최신, 최신)

# 파일 탐지 실행
base_dir = config.dataload_dir
available_files = find_mobile_plan_files(base_dir)

print(f"\n탐지된 mobile_plan_info 파일들:")
for date_str, filename in available_files:
    print(f"- {filename} (날짜: {date_str})")

previous_file, current_file = get_comparison_files(base_dir)
print(f"\n비교 대상:")
print(f"- 이전 파일: {previous_file}")
print(f"- 현재 파일: {current_file}")

## 3. 데이터 로드 및 구조 분석

In [ ]:
def load_json_file(file_path: str) -> Optional[Dict[str, Any]]:
    """JSON 파일 로드"""
    try:
        if not Path(file_path).exists():
            logger.warning(f"File not found: {file_path}")
            return None
        
        with open(file_path, 'r', encoding='utf-8') as f:
            return json.load(f)
    except Exception as e:
        logger.error(f"Error loading {file_path}: {str(e)}")
        return None

def extract_product_list(data: Dict[str, Any]) -> List[Dict[str, Any]]:
    """JSON 데이터에서 상품 리스트 추출"""
    # 새로운 형식 (metadata + result_list)
    if 'result_list' in data:
        return data['result_list']
    # 기존 형식 (직접 배열)
    elif isinstance(data, list):
        return data
    # 기타 형식
    else:
        logger.warning(f"Unknown data format: {list(data.keys()) if isinstance(data, dict) else type(data)}")
        return []

# 파일 로드
previous_data = None
current_data = None

if previous_file:
    previous_path = Path(base_dir) / previous_file
    previous_data = load_json_file(str(previous_path))
    if previous_data:
        previous_products = extract_product_list(previous_data)
        print(f"\n이전 데이터 로드 완료: {len(previous_products)}개 상품")
    else:
        print(f"\n이전 데이터 로드 실패: {previous_file}")
        previous_products = []
else:
    print(f"\n이전 데이터 없음 - 첫 번째 실행")
    previous_products = []

if current_file:
    current_path = Path(base_dir) / current_file
    current_data = load_json_file(str(current_path))
    if current_data:
        current_products = extract_product_list(current_data)
        print(f"현재 데이터 로드 완료: {len(current_products)}개 상품")
        
        # 메타데이터 확인
        if 'metadata' in current_data:
            print(f"현재 데이터 메타데이터: {current_data['metadata']}")
    else:
        print(f"현재 데이터 로드 실패: {current_file}")
        current_products = []
else:
    print(f"현재 데이터 없음")
    current_products = []

if not current_products:
    raise ValueError("현재 데이터가 없어서 비교할 수 없습니다.")

## 4. 상품 딕셔너리 변환 및 ID 매핑

In [ ]:
def get_product_dict(data: List[Dict[str, Any]]) -> Dict[str, Dict[str, Any]]:
    """상품 리스트를 상품 ID 기준 딕셔너리로 변환"""
    product_dict = {}
    extraction_stats = {"success": 0, "fallback": 0, "failed": 0}
    
    for product in data:
        product_id = None
        
        # 1차: mappedProductCode에서 상품 코드 추출
        try:
            mapped_code = product.get("managementInfo", {}).get("mappedProductCode", {})
            product_codes = mapped_code.get("productCode", {}).get("valueList", [])
            
            if product_codes and len(product_codes) > 0:
                product_id = str(product_codes[0])  # 첫 번째 코드 사용
                extraction_stats["success"] += 1
                
        except (KeyError, TypeError, AttributeError):
            pass
        
        # 2차: pmProductID 사용 (fallback)
        if not product_id:
            pm_id = product.get("pmProductID")
            if pm_id:
                product_id = str(pm_id)
                extraction_stats["fallback"] += 1
        
        # 3차: 기타 ID 필드들 시도
        if not product_id:
            for id_field in ["productId", "id", "productCode"]:
                if id_field in product and product[id_field]:
                    product_id = str(product[id_field])
                    extraction_stats["fallback"] += 1
                    break
        
        if product_id:
            product_dict[product_id] = product
        else:
            extraction_stats["failed"] += 1
            logger.warning(f"Product ID 추출 실패: {list(product.keys())[:5]}...")
    
    return product_dict, extraction_stats

# 상품 딕셔너리 변환
previous_products_dict, prev_stats = get_product_dict(previous_products)
current_products_dict, curr_stats = get_product_dict(current_products)

print(f"\n상품 ID 추출 결과:")
print(f"이전 데이터: 성공 {prev_stats['success']}, 대체 {prev_stats['fallback']}, 실패 {prev_stats['failed']}")
print(f"현재 데이터: 성공 {curr_stats['success']}, 대체 {curr_stats['fallback']}, 실패 {curr_stats['failed']}")

print(f"\n상품 ID 기준 딕셔너리 변환 완료:")
print(f"- 이전: {len(previous_products_dict)}개 상품")
print(f"- 현재: {len(current_products_dict)}개 상품")

if current_products_dict:
    sample_id = list(current_products_dict.keys())[0]
    print(f"\n현재 데이터 샘플 ID: {sample_id}")
    sample_product = current_products_dict[sample_id]
    print(f"샘플 상품 최상위 키: {list(sample_product.keys())[:10]}")

## 5. 깊은 비교 함수 정의 및 변경사항 감지

In [ ]:
def deep_compare(obj1: Any, obj2: Any, path: str = "") -> List[Dict[str, Any]]:
    """두 객체를 깊이 비교하여 새로 추가된 필드만 찾기"""
    changes = []
    
    # 타입이 다른 경우 추가로 간주하지 않음 (무시)
    if type(obj1) != type(obj2):
        return changes
    
    if isinstance(obj1, dict) and isinstance(obj2, dict):
        # 딕셔너리 비교 - 새로 추가된 키만 찾기
        old_keys = set(obj1.keys())
        new_keys = set(obj2.keys())
        
        # 새로 추가된 키들만 처리
        added_keys = new_keys - old_keys
        
        for key in added_keys:
            current_path = f"{path}.{key}" if path else key
            changes.append({
                "field": current_path,
                "change_type": "added",
                "old_value": None,
                "new_value": obj2[key]
            })
        
        # 공통 키들에 대해서는 재귀적으로 비교 (하위 필드의 추가 찾기)
        common_keys = old_keys & new_keys
        for key in common_keys:
            current_path = f"{path}.{key}" if path else key
            changes.extend(deep_compare(obj1[key], obj2[key], current_path))
    
    elif isinstance(obj1, list) and isinstance(obj2, list):
        # 리스트 비교 - 새로 추가된 항목만 찾기
        if len(obj2) > len(obj1):
            for i in range(len(obj1), len(obj2)):
                current_path = f"{path}[{i}]"
                changes.append({
                    "field": current_path,
                    "change_type": "added",
                    "old_value": None,
                    "new_value": obj2[i]
                })
        
        # 기존 항목들에 대해서는 재귀적으로 비교 (하위 필드 추가 찾기)
        min_len = min(len(obj1), len(obj2))
        for i in range(min_len):
            current_path = f"{path}[{i}]"
            changes.extend(deep_compare(obj1[i], obj2[i], current_path))
    
    return changes

# 변경사항 감지 시작
print(f"\n=== 변경사항 감지 시작 ===")

# ID 세트 분석
previous_ids = set(previous_products_dict.keys())
current_ids = set(current_products_dict.keys())

added_ids = current_ids - previous_ids
common_ids = previous_ids & current_ids
removed_ids = previous_ids - current_ids

print(f"ID 변화:")
print(f"- 새로 추가된 상품 ID: {len(added_ids)}개")
print(f"- 공통 상품 ID: {len(common_ids)}개")
print(f"- 제거된 상품 ID: {len(removed_ids)}개")

# 완전히 새로 추가된 상품들
completely_new_products = []
for product_id in added_ids:
    product_data = current_products_dict[product_id]
    # _product_id 추가 (변경사항 추적용)
    product_with_id = product_data.copy()
    product_with_id['_product_id'] = product_id
    completely_new_products.append(product_with_id)

print(f"\n완전히 새로운 상품 {len(completely_new_products)}개 처리 완료")
if completely_new_products:
    print(f"샘플: {completely_new_products[0]['_product_id']}")

## 6. 기존 상품에서 새 필드 감지

In [ ]:
# 기존 상품에서 새로 추가된 필드가 있는 상품들 찾기
products_with_new_fields = []
total_field_changes = 0

print(f"\n공통 상품 {len(common_ids)}개에서 새 필드 검사 중...")

processed_common = 0
for product_id in common_ids:
    processed_common += 1
    
    # 진행 상황 출력 (매 1000개마다)
    if processed_common % 1000 == 0 or processed_common == len(common_ids):
        print(f"진행: {processed_common}/{len(common_ids)} ({processed_common/len(common_ids)*100:.1f}%)")
    
    changes = deep_compare(
        previous_products_dict[product_id], 
        current_products_dict[product_id],
        f"product_{product_id}"
    )
    
    # added 타입의 변경사항이 있으면 포함
    if changes:
        products_with_new_fields.append({
            "product_id": product_id,
            "product_data": current_products_dict[product_id],
            "new_fields": changes  # 모든 변경사항이 이미 "added" 타입임
        })
        total_field_changes += len(changes)

print(f"\n새 필드가 추가된 상품: {len(products_with_new_fields)}개")
print(f"총 새 필드 변경사항: {total_field_changes}개")

# 샘플 출력
if products_with_new_fields:
    sample = products_with_new_fields[0]
    print(f"\n샘플 - 상품 {sample['product_id']}:")
    for field_change in sample['new_fields'][:3]:  # 처음 3개만
        print(f"  + {field_change['field']}: {str(field_change['new_value'])[:100]}...")

## 7. 변경사항 결과 정리

In [ ]:
# 전체 변경사항 결과 정리
changes_result = {
    "timestamp": datetime.now().isoformat(),
    "comparison_info": {
        "previous_file": previous_file,
        "current_file": current_file,
        "previous_products_count": len(previous_products_dict),
        "current_products_count": len(current_products_dict)
    },
    "change_summary": {
        "total_new_field_changes": len(completely_new_products) + len(products_with_new_fields),
        "completely_new_products": len(completely_new_products),
        "products_with_new_fields": len(products_with_new_fields),
        "total_field_additions": len(completely_new_products) + total_field_changes,
        "removed_products": len(removed_ids)
    },
    "changes": {
        "completely_new_products": completely_new_products,
        "products_with_new_fields": products_with_new_fields
    }
}

print(f"\n=== 변경사항 감지 결과 요약 ===")
print(f"비교 대상:")
print(f"- 이전: {previous_file} ({len(previous_products_dict)}개 상품)")
print(f"- 현재: {current_file} ({len(current_products_dict)}개 상품)")

print(f"\n변경사항:")
summary = changes_result['change_summary']
print(f"- 완전히 새로운 상품: {summary['completely_new_products']}개")
print(f"- 새 필드가 추가된 기존 상품: {summary['products_with_new_fields']}개")
print(f"- 총 새 필드 변경사항: {summary['total_new_field_changes']}개")
print(f"- 개별 필드 추가 수: {summary['total_field_additions']}개")

# 다음 단계로 전달할 변수
detected_changes = changes_result
has_changes = summary['total_new_field_changes'] > 0

print(f"\n변경사항 있음: {has_changes}")

## 8. 변경사항 파일로 저장

In [ ]:
# 변경사항이 있는 경우에만 저장
saved_change_files = []

if has_changes:
    if not version_date:
        version_date = datetime.now().strftime("%Y%m%d")
    
    # 1. JSON 형식 저장
    json_filename = f"field_changes_{version_date}.json"
    json_path = Path(config.dataload_dir) / json_filename
    
    try:
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(detected_changes, f, ensure_ascii=False, indent=2)
        print(f"\nJSON 저장 완료: {json_path}")
        saved_change_files.append(str(json_path))
    except Exception as e:
        logger.error(f"JSON 저장 실패: {str(e)}")
    
    # 2. 텍스트 요약 저장
    def create_text_summary(changes_data: Dict[str, Any]) -> str:
        """변경사항을 텍스트 요약으로 변환"""
        lines = []
        lines.append("=== 상품 데이터 변경사항 요약 ===")
        lines.append(f"타임스탬프: {changes_data.get('timestamp', 'N/A')}")
        lines.append("")
        
        summary = changes_data.get('change_summary', {})
        lines.append("■ 변경 요약:")
        lines.append(f"  - 전체 변경사항: {summary.get('total_new_field_changes', 0)}개")
        lines.append(f"  - 완전히 새로운 상품: {summary.get('completely_new_products', 0)}개")
        lines.append(f"  - 새 필드가 추가된 상품: {summary.get('products_with_new_fields', 0)}개")
        lines.append("")
        
        changes = changes_data.get('changes', {})
        
        # 완전히 새로운 상품
        new_products = changes.get('completely_new_products', [])
        if new_products:
            lines.append("■ 완전히 새로운 상품:")
            for product in new_products[:5]:  # 최대 5개만 표시
                product_id = product.get('_product_id') or product.get('product_id', 'Unknown')
                lines.append(f"  - {product_id}")
            if len(new_products) > 5:
                lines.append(f"  ... 외 {len(new_products) - 5}개")
            lines.append("")
        
        # 새 필드가 추가된 상품
        modified_products = changes.get('products_with_new_fields', [])
        if modified_products:
            lines.append("■ 새 필드가 추가된 상품:")
            for product in modified_products[:10]:  # 최대 10개만 표시
                product_id = product.get('product_id', 'Unknown')
                new_fields = product.get('new_fields', [])
                lines.append(f"  - {product_id} ({len(new_fields)}개 필드 추가)")
                for field in new_fields[:3]:  # 최대 3개 필드만 표시
                    field_name = field.get('field', 'Unknown')
                    field_value = str(field.get('new_value', 'N/A'))[:50] + '...' if len(str(field.get('new_value', 'N/A'))) > 50 else str(field.get('new_value', 'N/A'))
                    lines.append(f"    * {field_name}: {field_value}")
                if len(new_fields) > 3:
                    lines.append(f"    ... 외 {len(new_fields) - 3}개 필드")
            if len(modified_products) > 10:
                lines.append(f"  ... 외 {len(modified_products) - 10}개 상품")
        
        return "\n".join(lines)
    
    summary_text = create_text_summary(detected_changes)
    txt_filename = f"field_changes_summary_{version_date}.txt"
    txt_path = Path(config.dataload_dir) / txt_filename
    
    try:
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(summary_text)
        print(f"텍스트 요약 저장 완료: {txt_path}")
        saved_change_files.append(str(txt_path))
    except Exception as e:
        logger.error(f"텍스트 저장 실패: {str(e)}")
    
    print(f"\n총 {len(saved_change_files)}개 변경사항 파일 저장 완료")
    
else:
    print("\n변경사항이 없어서 파일을 저장하지 않습니다.")

# 다음 단계로 전달할 변수들
change_detection_result = {
    "changes": detected_changes,
    "has_changes": has_changes,
    "output_files": saved_change_files,
    "change_summary": changes_result['change_summary']
}

print(f"\n다음 단계로 전달할 정보:")
print(f"- 변경사항 유무: {has_changes}")
print(f"- 저장된 파일 수: {len(saved_change_files)}")
print(f"- 요약: {change_detection_result['change_summary']}")

## 완료

변경사항 감지가 완료되었습니다. 다음 단계에서 동료의 작업을 실행합니다.